# FDI tooth detector — train v2 on DENTEX (Colab)

**Run every cell top to bottom.** There is one decision to make, in Step 2, and the cell tells
you what to do. Nothing here needs editing otherwise.

Thin runner: all real code lives in the repo. Edit and push there, never paste code into this
notebook.

**Before you start: Runtime → Change runtime type → GPU → L4.**

### What v2 changes, and why

| | v1 (shipped as `best.pt`) | v2 (this notebook) | why |
|---|---|---|---|
| classes | 32, one per FDI code | **1 (`tooth`)** | ~18k boxes train one class instead of ~560 each. Numbering moves to `number_teeth.py`, which derives it from position in the arch. Also retires the rule that deleted a real tooth when two shared a code. |
| model | `yolov8n`, 3M params | **`yolov8s`**, 11M | nano was chosen to fit a free T4, not because it was right |
| train resolution | 1024 | **1536** (see Step 2) | teeth are small and densely packed; v1's missed teeth are what broke numbering |
| mosaic augmentation | 0.4 | **0.0** | mosaic collages four images together, destroying the whole-arch geometry that numbering depends on |

Everything writes to a **separate** Drive folder, so v1's dataset and `best.pt` are untouched
and every published number stays reproducible.

*(The v1 commands are kept for the record in the appendix at the bottom, as text. Don't run them.)*


In [ ]:
# Mount Drive and confirm which GPU you actually got.
from google.colab import drive; drive.mount('/content/drive')
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('GPU:', name or 'NONE — Runtime > Change runtime type > GPU > L4')
if name and 'T4' in name:
    print('\nNote: this is a T4, not an L4. It will work, but use the batch-8 cell in Step 3')
    print('and expect roughly double the training time.')

In [ ]:
# Pull the latest code, install deps, set paths. Re-run this after any push.
import os
if not os.path.exists('dental'):
    !git clone -q https://github.com/leostuy2028/dental.git
!cd dental && git pull -q
!pip install -q -r dental/detector/requirements.txt

DS = '/content/drive/MyDrive/dentex_yolo_1cls'   # v2 dataset + runs (separate from v1)
NAME = 'dentex_v2_1cls'
!cd dental && git log -1 --oneline
print('v2 dataset dir:', DS)

## Step 1 — build the single-class dataset

Streams DENTEX set (b), screens out the mis-counted images, writes YOLO labels with every
tooth as class 0.

**Expect:** `kept 618 clean images (557 train / 61 val)` and `EXCLUDED 16 ... duplicate_fdi`.

The verify line should say **"box counts match QC, all class 0"**. If it says anything about
duplicate classes instead, the `--single-class` flag didn't take — stop and report it.

If it prints **`VERIFY FAILED`**, stop. That means a mis-counted image reached the training
set, which is the one thing that would invalidate a tooth counter.

Takes a few minutes (it streams a 10.9 GB zip). Run once — skip it on later sessions.


In [ ]:
!python dental/detector/prepare_data.py --out {DS} --single-class

## Step 2 — the one decision: what resolution to train at

This reads DENTEX's **native** image size off Drive. It matters because training above the
data's real resolution just upscales empty pixels and wastes the GPU hour.

Read the recommendation the cell prints, then use it in Step 3.


In [ ]:
import glob
from collections import Counter
from PIL import Image
fs = glob.glob(f'{DS}/images/train/*')[:80]
assert fs, 'No images found — run Step 1 first.'
sizes = [Image.open(p).size for p in fs]
widths = sorted(w for w, _ in sizes)
med = widths[len(widths) // 2]
print('DENTEX native sizes, most common:', Counter(sizes).most_common(5))
print('median width:', med)
print()
if med >= 1800:
    IMGSZ = 1536
    print(f'-> Images are large (median {med}px). Train at IMGSZ = 1536. Nothing to change.')
else:
    IMGSZ = 1024
    print(f'-> Images are only {med}px wide. 1536 would upscale empty pixels.')
    print('   Training at IMGSZ = 1024 instead; the gain will come from the bigger')
    print('   model and mosaic being off.')
print(f'\nIMGSZ = {IMGSZ}  (Step 3 picks this up automatically)')

## Step 3 — train v2

Roughly 1–2 hours on an L4, longer on a T4. Progress prints per epoch; curves save to Drive.

`IMGSZ` comes from Step 2 — don't set it by hand.

**If this dies with CUDA out of memory**, run the batch-8 cell directly below it.


In [ ]:
!python dental/detector/train.py --data {DS}/dentex.yaml --project {DS}/runs \
    --model yolov8s.pt --name {NAME} --imgsz {IMGSZ} --batch 16 --epochs 150 --mosaic 0.0

**Only if the cell above ran out of memory.** Smaller batch, and it resumes from wherever
the failed run stopped.


In [ ]:
!python dental/detector/train.py --data {DS}/dentex.yaml --project {DS}/runs \
    --model yolov8s.pt --name {NAME} --imgsz {IMGSZ} --batch 8 --epochs 150 --mosaic 0.0 --resume

## Step 4 — validate on the held-out DENTEX split

**`COUNT physical` is the gate** — that's the number to compare against v1.

FDI and enum accuracy print as `n/a` here, deliberately: this model has one class, so there is
no FDI code for it to score. Numbering is scored separately in Step 5, against the benchmark's
own answers.

**v1's numbers to beat:** DENTEX val mAP50 0.935, exact count 51%, mean error 0.70.


In [ ]:
W2 = f'{DS}/runs/{NAME}/weights/best.pt'
!python dental/detector/validate.py --weights {W2} --data {DS}/dentex.yaml

In [ ]:
# Training curves and sample predictions
import os
from IPython.display import Image as IPImage, display
R = f'{DS}/runs/{NAME}'
for p in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    if os.path.exists(f'{R}/{p}'):
        print(p); display(IPImage(f'{R}/{p}', width=900))
print('\nweights:', W2)

## Step 5 — hand the weights back for the MMOral evaluation

The MMOral half runs locally on CPU. Download `best.pt` from the path printed above and save
it in your `dental` repo as **`best_v2.pt`** — *alongside* `best.pt`, never over it, since
every published number traces to v1's checkpoint.

Then locally:

```bash
python detector/tune_inference.py --weights best_v2.pt --raw results/detector/inference_raw_confs_v2.json --out results/detector/inference_sweep_v2.csv
```

```bash
python detector/eval_numbering.py --weights best_v2.pt --out results/detector/numbering_wisdom_v2.csv
```

**Paste back:** Step 4's output plus those two commands' output. That is everything needed to
decide whether v2 replaces v1.

What we're looking for: counting up from 51% exact, and wisdom-tooth numbering up from 37%
past the 40–48% the VLMs score. If numbering clears that, step 3 of the plan — the router —
has something worth routing to.


---
## Appendix — v1, for the record

These produced the committed `best.pt` and every detector number currently in the paper.
Kept as text on purpose, so they can't be run by accident and overwrite v2's work.

```
DATA = '/content/drive/MyDrive/dentex_yolo'
python dental/detector/prepare_data.py --out $DATA
python dental/detector/train.py --data $DATA/dentex.yaml --project $DATA/runs \
    --epochs 120 --imgsz 1024 --batch 8          # yolov8n, mosaic was 0.4
python dental/detector/validate.py --weights $DATA/runs/dentex_yolov8n/weights/best.pt \
    --data $DATA/dentex.yaml
```

Result: DENTEX val mAP50 0.935, exact count 51%, mean error 0.70; on MMOral 51% exact /
86% within one tooth / 0.87 mean error.

### SSL pretraining (Stage 1) — not part of v2

`pretrain_ssl.py` does contrastive pretraining on DENTEX's unlabeled panoramics. Skipped for
now because the measured failure is missed teeth in crowded rows, which resolution and
capacity address directly, and because its weights target a ResNet backbone rather than
YOLO's. Revisit if v2 does *not* fix the missed teeth — that would be evidence the model is
representation-limited after all, which is when SSL becomes well motivated rather than
speculative.


---
# STEP 4 — the DISEASE detector (optional, ~1 hr)

A second detector over DENTEX set (c), which labels four diagnoses: **Impacted, Caries, Deep
Caries, Periapical Lesion**. Composes with the tooth detector: the pathology box says *what*,
the positional numbering says *which tooth*.

Why it is worth a run — measured on gemini-3.5-flash's best config (62.7% overall):

| unrouted question type | n | gemini |
|---|---|---|
| periapical lesion | 81 | 53.1% |
| impacted / erupted | 51 | 54.9% |
| caries | 44 | 52.3% |
| **union** | **174** | **53.4%** (−9.3 vs its own average) |

That is 35% of the closed half sitting ~9 points below the model's average. If the detector
can answer even part of it, the routable set grows from 41 questions to a few hundred.

**Set expectations before reading the metrics.** The classes are badly imbalanced — Caries has
2,189 boxes, Periapical Lesion only 158 — and periapical lesions carry a known modality
ceiling (33% sensitivity on panoramic vs 78% on CBCT, paper ref [4]). Expect that class to
train worst, and do not read a poor result there as purely a model failure.

**Different QC from Step 2, deliberately.** Set (c) annotates only ABNORMAL teeth, so there is
no count invariant: a 3-box image is a mouth with 3 findings, and a repeated FDI code is a
tooth with two findings. The gate that protects the counter would throw away real data here.


In [ ]:
# S4.1  Build the disease dataset (own folder; leaves the tooth detectors alone)
DS_DIS = '/content/drive/MyDrive/dentex_disease'
!python dental/detector/prepare_disease.py --out {DS_DIS}

In [ ]:
# S4.2  Train.  Same recipe as the tooth detector: mosaic off, 1536, yolov8s.
NAME_DIS = 'dentex_disease_v1'
!python dental/detector/train.py --data {DS_DIS}/dentex.yaml --project {DS_DIS}/runs     --model yolov8s.pt --name {NAME_DIS} --imgsz 1536 --batch 16 --epochs 150 --mosaic 0.0

In [ ]:
# S4.3  NOT NEEDED — skip this cell.
#
# Ultralytics already validates at the end of training and prints the per-class table, which
# is the only readout that matters here. Re-running validate.py would repeat the mAP AND print
# COUNT/FDI lines that are meaningless for this model: that script computes a TOOTH count and
# FDI-code accuracy, and this detector's four classes are diagnoses, not teeth. A meaningless
# number that looks like a metric is worse than no number.
#
# Read the per-class mAP50 from the S4.2 output instead. Measured 2026-07-31:
#   Impacted           0.941   <- strong; this is the class worth routing
#   Caries             0.563
#   Deep Caries        0.534
#   Periapical Lesion  0.328   <- 158 training boxes AND a modality ceiling (ref [4])
print('skip S4.3 — read the per-class table printed by S4.2')

### S4.4 — bring it back

Download as **`best_disease.pt`** alongside `best.pt` and `best2.pt`. Then locally we compose
it with the tooth numbering and measure, per class, whether it beats gemini's 52–55% before
anything gets routed to it.
